## Demonstration notebook

### Day 1

- ➕ создание активного и замороженного счёта
- 🚫 попытка операций над замороженным счётом
- ✅ валидное пополнение и снятие

In [1]:
from datetime import datetime
from bank.accounts import BankAccount

day = datetime(2026, 11, 26, 15, 0)
account_active = BankAccount("12345678", "John Doe")
account_frozen = BankAccount(account_holder="Jane Smith", account_status="frozen")
account_active._clock = lambda: day
account_frozen._clock = lambda: day
print(account_active)
print(account_frozen)

BankAccount: Account holder: John Doe, Account ID: *5678, Status: active, Balance: 0.00 RUB
BankAccount: Account holder: Jane Smith, Account ID: *5103, Status: frozen, Balance: 0.00 RUB


In [2]:
try:
    account_frozen.deposit(100)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)

AccountFrozenError: Operation cannot be performed on a frozen account.


In [3]:
balance = account_active.deposit(200)
print(f"Deposited 200 {account_active.currency.value}. New balance: {balance} {account_active.currency.value}.")
balance = account_active.withdraw(100)
print(f"Withdrew 100 {account_active.currency.value}. New balance: {balance} {account_active.currency.value}.")

Deposited 200 RUB. New balance: 200.00 RUB.
Withdrew 100 RUB. New balance: 100.00 RUB.


### Day 2

Создать несколько счетов SavingsAccount, PremiumAccount, InvestmentAccount и выполнить различные операции

In [4]:
from bank.accounts import InvestmentAccount, PremiumAccount, SavingsAccount

savings = SavingsAccount(account_holder="Anna", interest_rate="0.01", min_balance="50")
premium = PremiumAccount(account_holder="Boris", overdraft_limit="300", commission_fixed="15")
investment = InvestmentAccount(account_holder="Clara")
for account in (savings, premium, investment):
    account._clock = lambda: day

In [5]:
balance = savings.deposit(200)
print(f"Deposited 200 {savings.currency.value}. New balance: {balance} {savings.currency.value}.")
interest = savings.apply_monthly_interest()
print(f"Applied monthly interest: {interest} {savings.currency.value}. New balance: {savings.balance} {savings.currency.value}.")
balance = savings.withdraw(100)
print(f"Withdrew 100 {savings.currency.value}. New balance: {balance} {savings.currency.value}.")
print(savings)

Deposited 200 RUB. New balance: 200.00 RUB.
Applied monthly interest: 2.0000 RUB. New balance: 202.0000 RUB.
Withdrew 100 RUB. New balance: 102.0000 RUB.
SavingsAccount: Account holder: Anna, Account ID: *5047, Status: active, Balance: 102.0000 RUB, Interest Rate: 1.00%, Minimum Balance: 50 RUB


In [6]:
balance = premium.deposit(100)
print(f"Deposited 100 {premium.currency.value}. New balance: {balance} {premium.currency.value}.")
balance = premium.withdraw(120)
print(f"Withdrew 120 {premium.currency.value}. New balance: {balance} {premium.currency.value}.")
print(premium)

Deposited 100 RUB. New balance: 100.00 RUB.
Withdrew 120 RUB. New balance: -35.00 RUB.
PremiumAccount: Account holder: Boris, Account ID: *2586, Status: active, Balance: -35.00 RUB, Overdraft Limit: 300 RUB, Commission Fixed: 1500%


In [7]:
balance = investment.deposit(1000)
print(f"Deposited 1000 {investment.currency.value}. New balance: {balance} {investment.currency.value}.")
balance = investment.buy_funds("stock", 500)
print(f"Bought 500 {investment.currency.value} of STOCK. Portfolio: {investment.get_account_info()['portfolio']}. New balance: {balance} {investment.currency.value}.")
balance = investment.buy_funds("bond", 200)
print(f"Bought 200 {investment.currency.value} of BOND. Portfolio: {investment.get_account_info()['portfolio']}. New balance: {balance} {investment.currency.value}.")
balance = investment.buy_funds("etf", 100)
print(f"Bought 100 {investment.currency.value} of ETF. Portfolio: {investment.get_account_info()['portfolio']}. New balance: {balance} {investment.currency.value}.")
balance = investment.sell_funds("etf", 40)
print(f"Sold 40 {investment.currency.value} of ETF. Portfolio: {investment.get_account_info()['portfolio']}. New balance: {balance} {investment.currency.value}.")
balance = investment.withdraw(50)
print(f"Withdrew 50 {investment.currency.value}. New balance: {balance} {investment.currency.value}.")
growth = investment.project_yearly_growth({"stock": 0.10, "bond": 0.04, "etf": 0.07})
print(f"Projected yearly growth: {growth}.")
print(investment)

Deposited 1000 RUB. New balance: 1000.00 RUB.
Bought 500 RUB of STOCK. Portfolio: {STOCK: 500.00, BOND: 0.00, ETF: 0.00}. New balance: 500.00 RUB.
Bought 200 RUB of BOND. Portfolio: {STOCK: 500.00, BOND: 200.00, ETF: 0.00}. New balance: 300.00 RUB.
Bought 100 RUB of ETF. Portfolio: {STOCK: 500.00, BOND: 200.00, ETF: 100.00}. New balance: 200.00 RUB.
Sold 40 RUB of ETF. Portfolio: {STOCK: 500.00, BOND: 200.00, ETF: 60.00}. New balance: 240.00 RUB.
Withdrew 50 RUB. New balance: 190.00 RUB.
Projected yearly growth: {STOCK: 50.00, BOND: 8.00, ETF: 4.20}.
InvestmentAccount: Account holder: Clara, Account ID: *4060, Status: active, Balance: 190.00 RUB, Portfolio: {STOCK: 500.00, BOND: 200.00, ETF: 60.00}


In [8]:
try:
    savings.withdraw(1000)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)

InsufficientFundsError: Withdrawal would breach minimum balance requirement. (Balance: 102.0000, Amount: 1000)


In [9]:
try:
    premium.withdraw(500)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)

InsufficientFundsError: Withdrawal would exceed overdraft limit. (Balance: -35.00, Amount: 515)


In [10]:
try:
    investment.buy_funds("crypto", 10)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)

InvalidOperationError: Unsupported investment type: 'crypto'. Expected: STOCK, BOND, ETF.


### Day 3

- клиент старше 18 лет и его счета
- заморозка счёта
- ночной запрет и пометка подозрительной операции
- три неверных пароля блокируют вход
- общий баланс и рейтинг клиентов

In [11]:
from bank.bank import Bank

bank = Bank(clock=lambda: day)
anna = bank.add_client("Anna Ivanova", "2000-01-01", {"phone": "+79990000000"}, "secret")
boris = bank.add_client("Boris Petrov", "1990-05-05", {"email": "boris@example.com"}, "pass")
print(anna)
print(boris)

savings = bank.open_account(anna.client_id, "savings", min_balance="50")
premium = bank.open_account(boris.client_id, "premium", overdraft_limit="300", commission_fixed="15")
investment = bank.open_account(anna.client_id, "investment")
print(bank.deposit(savings.account_id, 200))
print(bank.withdraw(savings.account_id, 100))
print(bank.deposit(premium.account_id, 100))
print(bank.withdraw(premium.account_id, 120))
print(bank.deposit(investment.account_id, 1000))
print(investment.buy_funds("stock", 500))
print(bank.get_total_balance(anna.client_id))
print(bank.get_clients_ranking())

Client: Anna Ivanova, ID: 56154798, Status: active, Age: 26, Accounts: 0
Client: Boris Petrov, ID: 82097032, Status: active, Age: 36, Accounts: 0
SavingsAccount: Account holder: Anna Ivanova, Account ID: *3170, Status: active, Balance: 200.00 RUB, Interest Rate: 1.00%, Minimum Balance: 50 RUB
SavingsAccount: Account holder: Anna Ivanova, Account ID: *3170, Status: active, Balance: 100.00 RUB, Interest Rate: 1.00%, Minimum Balance: 50 RUB
PremiumAccount: Account holder: Boris Petrov, Account ID: *9202, Status: active, Balance: 100.00 RUB, Overdraft Limit: 300 RUB, Commission Fixed: 1500%
PremiumAccount: Account holder: Boris Petrov, Account ID: *9202, Status: active, Balance: -35.00 RUB, Overdraft Limit: 300 RUB, Commission Fixed: 1500%
InvestmentAccount: Account holder: Anna Ivanova, Account ID: *2593, Status: active, Balance: 1000.00 RUB, Portfolio: {STOCK: 0.00, BOND: 0.00, ETF: 0.00}
500.00
{'RUB': Decimal('1100.00')}
[{'client_id': '56154798', 'fio': 'Anna Ivanova', 'totals': {'RUB

In [12]:
bank.freeze_account(savings.account_id)
try:
    bank.deposit(savings.account_id, 10)
except Exception as exc:
    print(type(exc).__name__, exc)
bank.unfreeze_account(savings.account_id)

bank._clock = lambda: datetime(2026, 11, 26, 3, 0)
try:
    bank.deposit(premium.account_id, 10)
except Exception as exc:
    print(type(exc).__name__, exc)
print(boris.suspicious_actions)
bank._clock = lambda: day

for _ in range(3):
    try:
        bank.authenticate_client(anna.client_id, "wrong")
    except Exception as exc:
        print(type(exc).__name__, exc)
print("Locked:", anna.is_locked)

AccountFrozenError Operation cannot be performed on a frozen account.
NightOperationError Operations are not allowed from 00:00 to 05:00.
[{'time': datetime.datetime(2026, 11, 27, 23, 5, 0, 786348), 'reason': 'night deposit on account 77809202'}]
AuthenticationError Wrong password.
AuthenticationError Wrong password.
ClientLockedError Client is locked after failed login attempts.
Locked: True


### Day 4

- 10 транзакций в очереди
- приоритет, отложенная операция и отмена
- комиссия за внешний перевод
- отказ для замороженного счёта и для минуса на обычном счёте

In [13]:
from datetime import datetime, timedelta
from bank.accounts import BankAccount, PremiumAccount
from bank.transactions import Transaction, TransactionProcessor, TransactionQueue, TransactionType

day = datetime(2026, 11, 26, 15, 0)
source = BankAccount("60000001", "Anna")
target = BankAccount("60000002", "Boris")
premium = PremiumAccount("60000003", "Clara", commission_fixed="0", overdraft_limit="200")
frozen = BankAccount("60000004", "Dmitry")
for account, balance in ((source, 5000), (target, 0), (premium, 0), (frozen, 300)):
    account._clock = lambda: day
    if balance:
        account.deposit(balance)
frozen.freeze()
accounts = {account.account_id: account for account in (source, target, premium, frozen)}

queue = TransactionQueue()
transactions = [
    Transaction(100, source, target, priority=1, transaction_id="00000001"),
    Transaction(250, source, premium, priority=4, transaction_id="00000002"),
    Transaction(80, source, target, priority=2, transaction_id="00000003"),
    Transaction(40, premium, target, priority=3, transaction_id="00000004"),
    Transaction(100, source, transaction_type=TransactionType.EXTERNAL_OUT, transaction_id="00000005"),
    Transaction(50, sender=None, receiver=target, transaction_type=TransactionType.EXTERNAL_IN, transaction_id="00000006"),
    Transaction(30, source, premium, priority=0, transaction_id="00000007"),
    Transaction(20, source, target, execute_at=day + timedelta(days=1), transaction_id="00000008"),
    Transaction(15, frozen, target, transaction_id="00000009"),
    Transaction(10, source, target, transaction_id="00000010"),
]
for transaction in transactions:
    queue.add(transaction)

processor = TransactionProcessor(accounts, clock=lambda: day)
processed = processor.process(queue)
for transaction in transactions:
    reason = f", reason: {transaction.reason}" if transaction.reason else ""
    print(f"{transaction.transaction_id} {transaction.status.value}, fee {transaction.fee}{reason}")
print("Processed:", processor.processed_ids)
print(source)
print(target)
print(premium)

00000001 completed, fee 0.00
00000002 completed, fee 0.00
00000003 completed, fee 0.00
00000004 completed, fee 0.00
00000005 completed, fee 1.00
00000006 completed, fee 0.50
00000007 completed, fee 0.00
00000008 deferred, fee 0.00
00000009 failed, fee 0.00, reason: Operation cannot be performed on a frozen account.
00000010 completed, fee 0.00
Processed: ['00000002', '00000004', '00000003', '00000001', '00000005', '00000006', '00000007', '00000009', '00000010']
BankAccount: Account holder: Anna, Account ID: *0001, Status: active, Balance: 4429.00 RUB
BankAccount: Account holder: Boris, Account ID: *0002, Status: active, Balance: 279.50 RUB
PremiumAccount: Account holder: Clara, Account ID: *0003, Status: active, Balance: 240.00 RUB, Overdraft Limit: 200 RUB, Commission Fixed: 0%


### Day 5

- обычный перевод проходит с низким риском
- крупная ночная операция блокируется
- частые переводы на новый счёт повышают риск
- отчёт: подозрительные операции, профиль клиента, статистика ошибок

In [14]:
from datetime import datetime, timedelta
from bank.accounts import BankAccount
from bank.audit import AuditLog, RiskAnalyzer
from bank.bank import Bank
from bank.transactions import Transaction, TransactionProcessor, TransactionQueue

day = datetime(2026, 11, 26, 15, 0)
night = datetime(2026, 11, 26, 2, 0)
log = AuditLog()
analyzer = RiskAnalyzer(large_amount="1000", frequent_limit=3, frequent_window=timedelta(minutes=10))

anna = BankAccount("70000001", "Anna")
boris = BankAccount("70000002", "Boris")
clara = BankAccount("70000003", "Clara")
for account, balance in ((anna, 5000), (boris, 0), (clara, 0)):
    account._client_id = "anna" if account is anna else account.holder.lower()
    account._clock = lambda: day
    if balance:
        account.deposit(balance)
accounts = {account.account_id: account for account in (anna, boris, clara)}

queue = TransactionQueue()
operations = [
    Transaction(100, anna, boris, created_at=day, transaction_id="00000001"),
    Transaction(120, anna, boris, created_at=day + timedelta(minutes=1), transaction_id="00000002"),
    Transaction(1500, anna, clara, created_at=night, transaction_id="00000003"),
]
for transaction in operations:
    queue.add(transaction)

processor = TransactionProcessor(accounts, clock=lambda: day, risk_analyzer=analyzer, audit_log=log)
processor.process(queue)
for transaction in operations:
    print(f"{transaction.transaction_id} {transaction.status.value}. {transaction.reason or 'completed'}")

print("Suspicious:")
for item in analyzer.suspicious_operations():
    print(item.transaction_id, item.level.value, ", ".join(item.signals))
print("Profile:", analyzer.client_risk_profile("anna"))
print("Errors:", log.error_statistics())

bank = Bank(clock=lambda: night, audit_log=log, risk_analyzer=analyzer)
client = bank.add_client("Anna Ivanova", "2000-01-01", {"phone": "+7999"}, "secret", client_id="anna-bank")
account = bank.open_account(client.client_id)
bank._clock = lambda: day
bank.deposit(account.account_id, 3000)
bank._clock = lambda: night
try:
    bank.withdraw(account.account_id, 2000)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)
print("Balance after blocked withdrawal:", account.balance)

00000001 completed. completed
00000002 completed. completed
00000003 failed. Risk signals: large amount, night operation.
Suspicious:
00000003 high large amount, night operation
Profile: {'client_id': 'anna', 'operations': 3, 'highest_risk': 'high', 'signals': {'large amount': 1, 'night operation': 1}, 'known_recipients': ['70000002']}
Errors: {'errors': 0, 'critical': 1, 'by_message': {'transaction 00000003: Risk signals: large amount, night operation.': 1}}
RiskBlockedError: Risk signals: large amount, night operation.
Balance after blocked withdrawal: 3000.00
